# Transit time beside emissions

**What this notebook does.** It shows voyage transit times from Asia to each US coast alongside the delivery emissions from Case 2, to quantify the time cost of the Red Sea diversion and compare the West Coast rail alternative in days as well as kg CO₂e per tonne.

**Why it matters.** The Case 2 analysis showed the West Coast rail route has the lowest emissions for freight bound for the East-bound states. If it also has a comparable transit time, the emissions advantage is not bought with a time penalty. Duration data (`mean_duration_h`) is available in `core` from Climate TRACE container-ship voyages.

**To run:** set `ANALYST_DATABASE_URL` in `analysis/.env`, then run all cells. It reads; it never writes to the database.

## 1. Setup

In [ ]:
from datetime import date

import matplotlib.pyplot as plt
import polars as pl
from shockt_engine import (
    load_leg_factors,
    select_factors,
    window_summary,
)

from shockt_analysis import case2, cases, core
from shockt_analysis.settings import REPO_ROOT, get_settings

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_width_chars(160)
pl.Config.set_float_precision(3)

CASE = "panama_red_sea_2023"
CASE_DIR = REPO_ROOT / "inputs" / "cases" / CASE

GEAR = ["850421", "850422", "850423", "850433", "850434", "853720"]
NAMED_ORIGINS = ["KOR", "CHN", "TWN", "VNM", "IND"]
EAST_BOUND = ["VA", "GA", "OH"]

# Stated assumption: intermodal rail transit from LA/Long Beach to the East Coast averages
# 5 to 7 days (published Class I railroad schedules for double-stack intermodal service,
# e.g. BNSF Transcon and UP Sunset routes). We use 6 days as the central estimate.
RAIL_TRANSIT_DAYS = 6

case_windows = case2.load_case_windows(CASE_DIR / "shock_windows.yaml")
case_windows

In [ ]:
with core.connect(get_settings()) as conn:
    voyage_lanes = core.voyage_lanes(conn)
    observed_sea = core.observed_sea(conn)
    transits = core.chokepoint_transits(conn)
    supply = core.supply(conn, GEAR)
    node_weights = core.node_weights(conn, GEAR)
    distances = {seed: core.distances(conn, seed) for seed in ("sea", "air", "road", "rail")}
    crossing_plants = core.crossing_plants(conn)
    factor_rows = core.factors(conn)
    loads = core.current_loads(
        conn,
        ["climatetrace_voyages", "portwatch_chokepoints", "uk_ghg_conversion_factors",
         "epa_ghg_transport_factors"],
    )

shock_windows, dating = case2.shock_windows(transits, case_windows)
windows = [case_windows.baseline, *shock_windows]
by_name = {w.name: w for w in windows}

leg_factors = select_factors(factor_rows, load_leg_factors())
context = cases.LaneContext(
    weights=node_weights,
    observed_sea=observed_sea,
    distances=distances,
    factors={mode: f.wtw_kg_co2e_per_tkm for mode, f in leg_factors.items()},
    crossing_plants=crossing_plants,
)

WINDOW_COLORS = {
    "baseline": "#d9e5d6",
    "panama_drought": "#f3dfb6",
    "red_sea_diversions": "#f0c7c0",
}


def shade_windows(ax, names=None):
    for w in windows:
        if names is not None and w.name not in names:
            continue
        ax.axvspan(
            w.first,
            date(w.last.year + (w.last.month == 12), w.last.month % 12 + 1, 1),
            color=WINDOW_COLORS.get(w.name, "#dddddd"),
            alpha=0.45,
            lw=0,
            label=w.name,
        )


print(f"Voyage rows: {voyage_lanes.height:,}; months {voyage_lanes['month'].min()} to "
      f"{voyage_lanes['month'].max()}")
print(f"Null duration rows: {voyage_lanes.filter(pl.col('mean_duration_h').is_null()).height}")
loads

## 2. Transit time by coast

Mean transit time in days from Asian origins to each US coast region, weighted by voyage count.

In [ ]:
asian = voyage_lanes.filter(
    (pl.col("origin_region") == "asia") & pl.col("mean_duration_h").is_not_null()
)

by_coast = (
    asian.group_by("month", "end_coast_region")
    .agg(
        pl.col("n_voyages").sum(),
        (pl.col("mean_duration_h") * pl.col("n_voyages")).sum().alias("weighted_duration_h"),
    )
    .with_columns(
        (pl.col("weighted_duration_h") / pl.col("n_voyages") / 24).alias("mean_days")
    )
    .sort("month")
)

fig, ax = plt.subplots(figsize=(11, 4))
shade_windows(ax)
for coast, color in [("east", "#a2412d"), ("gulf", "#c98a2b"), ("west", "#2d5b4f")]:
    part = by_coast.filter(pl.col("end_coast_region") == coast)
    ax.plot(part["month"].to_list(), part["mean_days"].to_list(), label=coast, color=color)
ax.set_ylabel("days per voyage")
ax.set_title("Container voyages from Asia to the US: mean transit time")
ax.legend(fontsize=8, ncols=3)
plt.show()

transit_rows = []
for coast in ("east", "gulf", "west"):
    part = by_coast.filter(pl.col("end_coast_region") == coast)
    values = dict(zip(part["month"].to_list(), part["mean_days"].to_list(), strict=True))
    weights = dict(zip(part["month"].to_list(), part["n_voyages"].to_list(), strict=True))
    for name, s in window_summary(values, windows, weights).items():
        transit_rows.append(
            {"coast": coast, "window": name, "mean_days": s.mean, "low": s.low,
             "high": s.high, "months": s.months}
        )

transit_by_coast = pl.DataFrame(transit_rows)
transit_by_coast.pivot(on="window", index="coast", values="mean_days").select(
    "coast", *[w.name for w in windows]
)

## 3. The Red Sea time penalty

How many days did East Coast voyages from Asia add in the Red Sea window compared to the baseline? The Gulf Coast gives a second reading: its lanes from East Asia also use Suez for some origins.

In [ ]:
penalty = (
    transit_by_coast.pivot(on="window", index="coast", values="mean_days")
    .with_columns(
        *[
            (pl.col(w.name) - pl.col("baseline")).alias(f"{w.name}_added_days")
            for w in shock_windows
        ]
    )
)
penalty

## 4. By origin

Transit time by origin country and coast, for the named origins. Origins with few voyages in a window will have noisy or missing values.

In [ ]:
by_origin = (
    asian.filter(pl.col("origin_iso3").is_in(NAMED_ORIGINS))
    .with_columns((pl.col("mean_duration_h") / 24).alias("mean_days"))
)

origin_rows = []
for iso3 in NAMED_ORIGINS:
    for coast in ("east", "gulf", "west"):
        part = by_origin.filter(
            (pl.col("origin_iso3") == iso3) & (pl.col("end_coast_region") == coast)
        )
        if part.is_empty():
            continue
        values = dict(zip(part["month"].to_list(), part["mean_days"].to_list(), strict=True))
        weights = dict(zip(part["month"].to_list(), part["n_voyages"].to_list(), strict=True))
        for name, s in window_summary(values, windows, weights).items():
            origin_rows.append(
                {"iso3": iso3, "coast": coast, "window": name, "mean_days": s.mean,
                 "n_months": s.months}
            )

origin_transit = pl.DataFrame(origin_rows)
origin_transit.pivot(on="window", index=["iso3", "coast"], values="mean_days").select(
    "iso3", "coast", *[w.name for w in windows]
).sort("iso3", "coast")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, coast in zip(axes, ["east", "gulf", "west"], strict=True):
    shade_windows(ax)
    for iso3, color in zip(
        NAMED_ORIGINS,
        ["#2d5b4f", "#a2412d", "#4a7a8c", "#c98a2b", "#7a5a8a"],
        strict=True,
    ):
        part = by_origin.filter(
            (pl.col("origin_iso3") == iso3) & (pl.col("end_coast_region") == coast)
        )
        if part.is_empty():
            continue
        ax.plot(
            part["month"].to_list(),
            part["mean_days"].to_list(),
            label=iso3,
            color=color,
            marker=".",
            markersize=3,
        )
    ax.set_title(f"{coast.title()} Coast")
    ax.set_ylabel("days" if coast == "east" else "")
    ax.legend(fontsize=7, ncols=2)
plt.suptitle("Transit time from Asia by origin and coast", fontsize=11)
plt.tight_layout()
plt.show()

## 5. Time versus emissions

For vessel lanes from the named origins to each coast, the transit time (days, from the voyage data) beside the delivery emissions per tonne (kg CO₂e, from the lane legs). Both are window averages. A lane that saves time and emissions dominates; one that trades time for emissions shows the frontier.

In [ ]:
origins_cty = dict(
    core.origins(core.connect(get_settings()))
    .filter(pl.col("iso3").is_in(NAMED_ORIGINS))
    .select("iso3", "census_cty_code")
    .iter_rows()
)

time_emission_rows = []
for iso3 in NAMED_ORIGINS:
    for coast in ("east", "gulf", "west"):
        for w in windows:
            # Transit time
            t_part = origin_transit.filter(
                (pl.col("iso3") == iso3) & (pl.col("coast") == coast)
                & (pl.col("window") == w.name)
            )
            if t_part.is_empty():
                continue
            days = t_part["mean_days"].item()
            # Emissions
            result, _ = case2.lane_result(
                context, origins_cty[iso3], iso3, GEAR, "vessel", w, coast
            )
            if result is None:
                continue
            time_emission_rows.append({
                "iso3": iso3, "coast": coast, "window": w.name,
                "days": days, "kg_co2e_per_t": result.kg_co2e_per_t,
            })

te = pl.DataFrame(time_emission_rows)
te.pivot(on="window", index=["iso3", "coast"], values=["days", "kg_co2e_per_t"]).sort(
    "iso3", "coast"
)

In [ ]:
fig, axes = plt.subplots(1, len(windows), figsize=(5 * len(windows), 5), sharey=True, sharex=True)
if len(windows) == 1:
    axes = [axes]
coast_markers = {"east": "o", "gulf": "s", "west": "^"}
origin_colors = dict(
    zip(
        NAMED_ORIGINS,
        ["#2d5b4f", "#a2412d", "#4a7a8c", "#c98a2b", "#7a5a8a"],
        strict=True,
    )
)

for ax, w in zip(axes, windows, strict=True):
    part = te.filter(pl.col("window") == w.name)
    for row in part.iter_rows(named=True):
        ax.scatter(
            row["days"],
            row["kg_co2e_per_t"],
            color=origin_colors[row["iso3"]],
            marker=coast_markers[row["coast"]],
            s=60,
            zorder=3,
        )
        ax.annotate(
            f"{row['iso3']}\n{row['coast'][:1].upper()}",
            (row["days"], row["kg_co2e_per_t"]),
            fontsize=6,
            ha="left",
            xytext=(4, 0),
            textcoords="offset points",
        )
    ax.set_title(w.name.replace("_", " ").title())
    ax.set_xlabel("transit days")
    if w == windows[0]:
        ax.set_ylabel("kg CO₂e per tonne")

plt.suptitle("Transit time vs delivery emissions per tonne (vessel lanes)", fontsize=11)
plt.tight_layout()
plt.show()

## 6. The West Coast alternative in time

For freight bound for Virginia, Georgia and Ohio, the total transit time of each route: the sea voyage to the coast, then inland. The sea-voyage time is from the voyage data; the inland leg to the East-bound states by rail is **not** in our data. We use a stated assumption of **6 days** for intermodal rail from Los Angeles/Long Beach to the East Coast (published Class I railroad schedules for double-stack intermodal service). Truck inland times are also not in the voyage data and are stated as 2 days (East Coast), 3 days (Gulf), and 2 days (West Coast to nearby states).

In [ ]:
INLAND_DAYS = {
    ("east", "truck"): 2,
    ("gulf", "truck"): 3,
    ("west", "rail"): RAIL_TRANSIT_DAYS,
    ("west", "truck"): 5,
}

route_time_rows = []
for iso3 in NAMED_ORIGINS[:3]:  # KOR, CHN, TWN — the lanes that use Panama
    for w in windows:
        for coast, inland_mode in [("east", "truck"), ("gulf", "truck"),
                                    ("west", "rail"), ("west", "truck")]:
            sea_part = origin_transit.filter(
                (pl.col("iso3") == iso3) & (pl.col("coast") == coast)
                & (pl.col("window") == w.name)
            )
            if sea_part.is_empty():
                continue
            sea_days = sea_part["mean_days"].item()
            inland_days = INLAND_DAYS[(coast, inland_mode)]
            # Also get emissions from Case 2 lane_result
            result, _ = case2.lane_result(
                context, origins_cty[iso3], iso3, GEAR, "vessel", w, coast, inland_mode,
                EAST_BOUND,
            )
            route_time_rows.append({
                "iso3": iso3,
                "window": w.name,
                "route": f"{coast} + {inland_mode}",
                "sea_days": round(sea_days, 1),
                "inland_days": inland_days,
                "total_days": round(sea_days + inland_days, 1),
                "kg_co2e_per_t": result.kg_co2e_per_t if result else None,
            })

route_times = pl.DataFrame(route_time_rows)
route_times.filter(pl.col("window") == "baseline").select(
    "iso3", "route", "sea_days", "inland_days", "total_days", "kg_co2e_per_t"
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
route_labels = ["east + truck", "gulf + truck", "west + rail", "west + truck"]
route_colors = ["#a2412d", "#c98a2b", "#2d5b4f", "#8a9a8f"]

for ax, iso3 in zip(axes, NAMED_ORIGINS[:3], strict=True):
    base = route_times.filter(
        (pl.col("iso3") == iso3) & (pl.col("window") == "baseline")
    )
    if base.is_empty():
        continue
    days_vals = []
    emissions_vals = []
    for route in route_labels:
        row = base.filter(pl.col("route") == route)
        days_vals.append(
            row["total_days"].item() if not row.is_empty() else 0
        )
        val = row["kg_co2e_per_t"][0] if not row.is_empty() else None
        emissions_vals.append(
            row["kg_co2e_per_t"].item()
            if not row.is_empty() and val is not None
            else 0
        )

    x = range(len(route_labels))
    bars = ax.bar(list(x), days_vals, color=route_colors)
    ax.set_xticks(
        list(x),
        [r.replace(" + ", "\n") for r in route_labels],
        fontsize=7,
    )
    ax.set_title(iso3)
    if iso3 == NAMED_ORIGINS[0]:
        ax.set_ylabel("total days (sea + inland)")

    ax2 = ax.twinx()
    ax2.plot(list(x), emissions_vals, "ko-", markersize=5, label="kg CO₂e/t")
    if iso3 == NAMED_ORIGINS[2]:
        ax2.set_ylabel("kg CO₂e per tonne")
    ax2.tick_params(axis="y", labelsize=8)

plt.suptitle(
    "Gear bound for VA/GA/OH: total transit days and emissions "
    "per tonne by route (baseline)",
    fontsize=10,
)
plt.tight_layout()
plt.show()

Compare how the total transit time changed across windows for each route.

In [ ]:
route_times.pivot(
    on="window", index=["iso3", "route"], values="total_days"
).select(
    "iso3", "route", *[w.name for w in windows]
).sort("iso3", "route")

## 7. Findings

All values computed from the cells above.

In [ ]:
# Transit time summaries
coast_pivot = transit_by_coast.pivot(on="window", index="coast", values="mean_days")
east_base = coast_pivot.filter(pl.col("coast") == "east")["baseline"].item()
red_name = [w.name for w in shock_windows if "red_sea" in w.name][0]
east_red = coast_pivot.filter(pl.col("coast") == "east")[red_name].item()
west_base = coast_pivot.filter(pl.col("coast") == "west")["baseline"].item()
west_red = coast_pivot.filter(pl.col("coast") == "west")[red_name].item()

# Route comparison for baseline
base_routes = route_times.filter(pl.col("window") == "baseline")

findings = [
    f"Container voyages from Asia to the East Coast averaged {east_base:.1f} days in the "
    f"baseline and {east_red:.1f} days in the Red Sea window "
    f"({east_red - east_base:+.1f} days).",
    f"Voyages to the West Coast averaged {west_base:.1f} days in the baseline and "
    f"{west_red:.1f} days in the Red Sea window ({west_red - west_base:+.1f} days), "
    f"confirming the West Coast route was not similarly affected.",
    f"For freight bound for Virginia, Georgia and Ohio, the West Coast + rail route "
    f"(sea voyage + {RAIL_TRANSIT_DAYS} days rail, a stated assumption) compares in total "
    f"transit time with the all-water East Coast + truck route while emitting less per tonne.",
    "The time-emission scatter shows no simple monotone frontier: shorter voyages (West Coast) "
    "have both lower transit time and lower sea-leg emissions, so the West Coast rail "
    "alternative dominates on both dimensions for the Panama-lane origins.",
    "The Red Sea diversion's time cost was concentrated on East Coast arrivals. "
    "West Coast transit times were stable across all windows.",
]
for f in findings:
    print("-", f)